# ENERGY CONSUMPTION PREDICTION

The Project is the prediction of the Power Consuption of the whole country depending on the county.The consumption determines the prices, The most users and To determine the Profits of the Company. In this project, I used Machine Learning Algorithms to determine the future and gowth of the energy sector.

In [ ]:
#importing necessary libraries

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib as plt
import matplotlib.pyplot as plt
import matplotlib.pylab as plt
import panel as pn
plt.style.use('ggplot')
import os
import tarfile
from six.moves import urllib
pn.extension('tabulator')
import lightgbm as lgbm

import hvplot.pandas

%matplotlib inline

In [ ]:
#Loading dataset

df = pd.read_excel("KplcData/Daset.xlsx") 
df


In [ ]:
#Basic length and shape of the data 

print("Dataset Length: ", len(df)) 
print("Dataset Shape: ", df.shape)

This describes the lenghth and shape of the data to check if they align with dataset provided 

In [ ]:
#Information of the data

df.info()

From .info(), we can tell that our data contains 201408 entries. The features are all both in numerical and object form in this csv file. From the dataset presented in .head(), the features are repeated and therefore, it is a categorical attribute. the attribute 'UNITS' has a total of 201267 features insteasd of 201408 features, therefore, some disricts are missing features. 

In [ ]:
#Top dataframe

df.head()

df.head() gives us a snippet of how our code should look like instaed of the whole dataset.

In [ ]:
#Columns of the dataframe 

df.columns

# Data Cleaning

Here we are creating a few function sto take care of missing values, get information on the dataset, transform dataset. Most machine learning algorithms cannot work with missing features. One can:

   - get rid of the corresponding districts (dropna())
   - get rid of the whole attribute. (drop())
   - set the vaues to some value ie zero, mean, median etc (fillna())


In [ ]:
#The statistics on the data

df.describe()

This reveals the mean, minimum, maximum and standard deviation of the numerical data. 25, 50 and 75 are the percentiles of each value.

In [ ]:
tariff_group = df.ACC.value_counts()
tariff_group = df.groupby(['COD_TARIFF', 'REGION']).sum()
tariff_group

-By using groupby, we find the value of each tariff in terms of their total units of all the meters used in the consumption of energy

In [ ]:
#Getting the days and months of installation of the smart meters 

df["year"] = pd.to_datetime(df["INSTALLATION_DATE"]).dt.year
df["month"] = pd.to_datetime(df["INSTALLATION_DATE"]).dt.month
df["day"] = pd.to_datetime(df["INSTALLATION_DATE"]).dt.day
df

In [ ]:
df.query('COUNTY == "MOMBASA"').describe()

In [ ]:
#Sum of the total number of null values in each column of the dataset 

df.isnull().sum()


In [ ]:
#Removing all the null values in the dataset and replacing them with 1.

df.fillna(1, inplace=True)

In [ ]:
df.isnull().sum()

In [ ]:
df.columns

In [ ]:
df["COUNTY"].unique()

In [ ]:
print(df['REGION'].value_counts())
print(df['COUNTY'].value_counts())
print(df['ACC'].value_counts())
print(df['COD_TARIFF'].value_counts())


# Data Vizualisation

In [ ]:
#To understand the relationship between different attributes in the dataset, we will plot a correlation matrix using the following code:

corrMatrix = df.corr() 
sns.heatmap(corrMatrix, annot=True) 
plt.show() 

In [ ]:
acc = df['ACC'].value_counts()
tariff_type = df['COD_TARIFF'].value_counts()
units= df['UNITS'].sum()
county = df['COUNTY'].value_counts()

In [ ]:
#Pie plot for all the tariffs used 


tariff_counts = df['COD_TARIFF'].value_counts(7)
tariff_counts

plt.figure(figsize=(20,6))
plt.title('Tariff Distribution')
plt.pie(tariff_counts, labels=tariff_counts.index, autopct='%1.1f%%' ,startangle=200, shadow=True);



In [ ]:
df['COD_TARIFF'].value_counts() \
   .head(10) \
   .plot(kind = 'bar')

In [ ]:
#Counties that consume the most energy by the total units consumed.

top_10_counties= df.COUNTY.value_counts().sort_values(ascending=False).head(7)
top_10_counties

#Plot for the top 10 counties

plt.figure(figsize=(12,10))
#plt.xticks(rotation=20)
plt.title('Overall Consumption by County')
sns.barplot(x=top_10_counties.index, y=top_10_counties, palette='Set2');

# CHANGING THE CATEGORICAL  DATA 

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from sklearn.preprocessing import LabelEncoder

LabelEncoder presents a problem where an ML algorithm will assume that two nearby values are more similar than two distant values. To prevent this, a method of encoding kmown as one hot encoding is used where only one category presents a one and 0 otherwise. It converts integer categorical values into on-hot vectors. fit_transform() expects a 2D array but house_cat_encoded is a 1D array and therefore needs to be reshaped.

In [ ]:
#encoding Columns 

df.replace({'COD_TARIFF':{
"A1-1 SC2"     :0,
"A1-1 SC1"     :1,
"E0-1"         :2,
"A0-1 DC-L"    :3,
"A0-1 DC-O"    :4,
"A1-1"         :5,
"A0-1"         :6,
"A0-2 DC-O"    :7,
"A0-2 DC-L"    :8,
"A1-2 SC2"     :9,
"A0-2"         :10,
"A1-2"         :11,
"A1-2 SC1"     :12,
"P0-1 DC-O"    :13,
"A1-5"         :14,
"A1-5 SC"      :15,
"P1-1 SC"      :16,
"P0-1 DC-L"    :17}},inplace=True) 

In [ ]:
#encoding categorical data 

from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
le = LabelEncoder()
X= df.iloc[:, 5:6].values  
X[:, 3] = le.fit_transform(X[:, 3])

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
ohe=ColumnTransformer([('OHE',OneHotEncoder(categories='auto'),['REGION'])],remainder='passthrough')
x=ohe.fit_transform(df.drop('COD_TARIFF',axis='columns'))
print(x)


In [ ]:
#encoding Columns 

df.replace({'COD_TARIFF':{
"A1-1 SC2"     :0,
"A1-1 SC1"     :1,
"E0-1"         :2,
"A0-1 DC-L"    :3,
"A0-1 DC-O"    :4,
"A1-1"         :5,
"A0-1"         :6,
"A0-2 DC-O"    :7,
"A0-2 DC-L"    :8,
"A1-2 SC2"     :9,
"A0-2"         :10,
"A1-2"         :11,
"A1-2 SC1"     :12,
"P0-1 DC-O"    :13,
"A1-5"         :14,
"A1-5 SC"      :15,
"P1-1 SC"      :16,
"P0-1 DC-L"    :17}},inplace=True) 


In [ ]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder

ohe=ColumnTransformer([('OHE',OneHotEncoder(categories='auto'),['REGION'])],remainder='passthrough')
x=ohe.fit_transform(df[['REGION']])


In [ ]:
df_encoded = df.values.reshape(-1,1) 
df_encoded

In [ ]:
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()
regions = df["REGION"]
regions_encoded = encoder.fit_transform(housing_cat)
regions_encoded



In [ ]:
print(encoder.classes_)


LabelEncoder presents a problem where an ML algorithm will assume that two nearby values are more similar than two distant values. To prevent this, a method of encoding kmown as one hot encoding is used where only one category presents a one and 0 otherwise. It converts integer categorical values into on-hot vectors. fit_transform() expects a 2D array but house_cat_encoded is a 1D array and therefore needs to be reshaped.

In [ ]:
regions_encoded = df.values.reshape(-1,1) 
regions_encoded

In [ ]:
#dropping the first column to avoid dummy variable trap

x=x[:,1:]
x


# SPLITTING THE DATASET

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin

class DataFrameSelector(BaseEstimator, TransformerMixin):
    def __init__(self, attribute_names):
        self.attribute_names = attribute_names
    def fit(self, X, y=None):
        return self
    def transform(self, X, y=None):
        return X[self.attribute_names].values

In [ ]:
df.isnull().sum()

In [ ]:
#Identifying the independent variable

X= df.iloc[:, 5:6].values  
X

In [ ]:
#Identifying the dependent variable

Y = df.iloc[:, 7:8].values 
Y

In [ ]:
X = df.iloc[:, 5:6].values 
Y = df.iloc[:, 7:8].values 


In [ ]:
#Separating dataset


from sklearn.model_selection import train_test_split
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size = 0.2, random_state=42) 

In [ ]:
X.shape, Y.shape

In [ ]:
X_train.shape, Y_train.shape

In [ ]:
X_test.shape, Y_test.shape

In [ ]:
#check the requirements of the model we are building to see if the data needs to be scaled.

from sklearn.preprocessing import StandardScaler
standardized = StandardScaler().fit_transform(X_train)
standardized[~np.isnan(standardized)][:30]

In [ ]:
# preprocessing module for min-max scaling (to normalize data to be in [0, 1]), among others.

from sklearn.preprocessing import MinMaxScaler
MinMaxScaler().fit_transform(X_train)


In [ ]:
#Fitting multiple linear regression to training set 

from sklearn.linear_model import LinearRegression
model_fit = LinearRegression()
model_fit.fit(X_train, Y_train)

In [ ]:
# loading the linear regression model 
lin_reg_model = LinearRegression() 

# Now we can fit the model to our dataset 
lin_reg_model.fit(X_train,Y_train) 

In [ ]:
#inspect the intercept 
print('Intercepts: ', lin_reg_model.intercept_)


In [ ]:
#inspect the coefficient
print('Coefficients: ', lin_reg_model.coef_)

In [ ]:
# variance score: 1 means perfect prediction
print('Variance score: {}'.format(lin_reg_model.score(X_test, Y_test)))
  


In [ ]:
# plot for residual error
  
## setting plot style
plt.style.use('fivethirtyeight')
  
## plotting residual errors in training data
plt.scatter(lin_reg_model.predict(X_train), lin_reg_model.predict(X_train) - Y_train,
            color = "red", s = 10, label = 'Train data')
  
## plotting residual errors in test data
plt.scatter(lin_reg_model.predict(X_test), lin_reg_model.predict(X_test) - Y_test,
            color = "blue", s = 10, label = 'Test data')
  
## plotting line for zero residual error
plt.hlines(y = 0, xmin = 0, xmax = 50, linewidth = 2)
  
## plotting legend
plt.legend(loc = 'upper right')
  
## plot title
plt.title("Residual errors")
  
## method call for showing the plot
plt.show()